In [ ]:
"""Colab-friendly feature distribution script for a CSV file.

Edit the CONFIG section below, then run the notebook cell.

What it does:
- Loads a CSV file with pandas.
- Selects numeric columns by default.
- Detects skewed features automatically.
- Uses linear, log, or symlog horizontal scale as appropriate.
- Exports high-resolution PNGs with 2 rows x 3 columns per image.
"""

from __future__ import annotations

from pathlib import Path
from typing import Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================
# CONFIG — edit these only
# =========================
CSV_PATH = "/content/all_59k_accidents_archive_weather_poi_final.csv"  # path to your CSV in Colab
OUTPUT_DIR = "/content/feature_distribution_plots"
COLUMNS = None  # e.g. ["age", "income", "score"], or None for all numeric columns
SKEW_THRESHOLD = 1.0
BINS = 30
DPI = 300
FIGSIZE = (18, 10)  # 2 rows x 3 cols
CLIP_QUANTILES = (0.01, 0.99)  # set to (0.0, 1.0) to disable clipping on linear plots

COLUMN_LABELS = {
    'year': 'Year',
    'collision_id': 'Collision ID',
    'borough': 'Borough',
    'easting': 'Easting',
    'northing': 'Northing',
    'casualty_ref': 'Casualty reference',
    'casualtyclass': 'Casualty class',
    'casualty_sex': 'Casualty sex',
    'age': 'Age',
    'casualty_count': 'Casualty count',
    'casualty_severity': 'Casualty severity',
    'mode_of_travel': 'Mode of travel',
    'vehicletype': 'Vehicle type',
    'vehicletypebanded': 'Vehicle type (banded)',
    'vehiclemanoeuvres': 'Vehicle manoeuvre',
    'vehicleskidding': 'Vehicle skidding',
    'restrictedlane': 'Restricted lane',
    'junctionlocation': 'Junction location',
    'objectincw': 'Object in carriageway',
    'vehoffcw': 'Vehicle off carriageway',
    'vehimpact': 'Vehicle impact',
    'journeypurpose': 'Journey purpose',
    'driversex': 'Driver sex',
    'location': 'Location',
    'noofcasualtiesinacc': 'Number of casualties in accident',
    'noofvehiclesinacc': 'Number of vehicles in accident',
    'accidentdate': 'Accident date',
    'day': 'Day',
    'time': 'Time',
    'highway': 'Highway authority',
    'roadclass1': 'Road class 1',
    'roadno1': 'Road number 1',
    'roadtype': 'Road type',
    'speedlimit': 'Speed limit (mph)',
    'junctiondetail': 'Junction detail',
    'junctioncontrol': 'Junction control',
    'roadclass2': 'Road class 2',
    'roadno2': 'Road number 2',
    'pedcrossingdecoded': 'Pedestrian crossing facility',
    'lightconditionsbanded': 'Light conditions (banded)',
    'weather': 'Weather',
    'roadsurface': 'Road surface',
    'specialconditions': 'Special conditions',
    'cwhazard': 'Carriageway hazard',
    'longitude': 'Longitude',
    'latitude': 'Latitude',
    'nearest_pub_crow_name': 'Nearest pub name',
    'nearest_pub_crow_fhrsid': 'Nearest pub FHRSID',
    'nearest_pub_crow_lat': 'Nearest pub latitude',
    'nearest_pub_crow_lon': 'Nearest pub longitude',
    'nearest_pub_crow_m': 'Crow-flight distance to nearest pub (m)',
    'nearest_school_crow_name': 'Nearest school name',
    'nearest_school_crow_type': 'Nearest school type',
    'nearest_school_crow_lat': 'Nearest school latitude',
    'nearest_school_crow_lon': 'Nearest school longitude',
    'nearest_school_crow_m': 'Crow-flight distance to nearest school (m)',
    'nearest_overground_crow_name': 'Nearest overground station name',
    'nearest_overground_crow_lat': 'Nearest overground station latitude',
    'nearest_overground_crow_lon': 'Nearest overground station longitude',
    'nearest_overground_crow_m': 'Crow-flight distance to nearest overground station (m)',
    'nearest_underground_crow_name': 'Nearest underground station name',
    'nearest_underground_crow_lat': 'Nearest underground station latitude',
    'nearest_underground_crow_lon': 'Nearest underground station longitude',
    'nearest_underground_crow_m': 'Crow-flight distance to nearest underground station (m)',
    'nearest_signal_crow_name': 'Nearest signal name',
    'nearest_signal_crow_lat': 'Nearest signal latitude',
    'nearest_signal_crow_lon': 'Nearest signal longitude',
    'nearest_signal_crow_m': 'Crow-flight distance to nearest signal (m)',
    'n_other_vehicles': 'Number of other vehicles',
    'other_vehicle_1': 'Other vehicle 1',
    'other_vehicle_2': 'Other vehicle 2',
    'other_vehicle_3': 'Other vehicle 3',
    'exposure_avg_cyclists': 'Average cyclist exposure',
    'n_sites_in_pool': 'Number of sites in pool',
    'match_level': 'Match level',
    'lightconditions_collapsed': 'Light conditions (collapsed)',
    'vehicle_manoeuvres_simple': 'Vehicle manoeuvre (simple)',
    'row_index': 'Row index',
    'accident_time_utc': 'Accident time (UTC)',
    'archive_retrieved_latitude': 'Archive retrieved latitude',
    'archive_retrieved_longitude': 'Archive retrieved longitude',
    'archive_hour_match_gap_seconds': 'Archive hour match gap (seconds)',
    'archive.hourly.temperature_2m': 'Hourly 2 m air temperature',
    'archive.hourly.temperature_2m_lag_1h': 'Hourly 2 m air temperature (1-hour lag)',
    'archive.hourly.temperature_2m_lag_3h': 'Hourly 2 m air temperature (3-hour lag)',
    'archive.hourly.temperature_2m_avg_last_3h': 'Hourly 2 m air temperature (3-hour rolling average)',
    'archive.hourly.temperature_2m_avg_last_6h': 'Hourly 2 m air temperature (6-hour rolling average)',
    'archive.hourly.relativehumidity_2m': 'Hourly 2 m relative humidity',
    'archive.hourly.relativehumidity_2m_lag_1h': 'Hourly 2 m relative humidity (1-hour lag)',
    'archive.hourly.relativehumidity_2m_lag_3h': 'Hourly 2 m relative humidity (3-hour lag)',
    'archive.hourly.relativehumidity_2m_avg_last_3h': 'Hourly 2 m relative humidity (3-hour rolling average)',
    'archive.hourly.relativehumidity_2m_avg_last_6h': 'Hourly 2 m relative humidity (6-hour rolling average)',
    'archive.hourly.dewpoint_2m': 'Hourly 2 m dew point',
    'archive.hourly.dewpoint_2m_lag_1h': 'Hourly 2 m dew point (1-hour lag)',
    'archive.hourly.dewpoint_2m_lag_3h': 'Hourly 2 m dew point (3-hour lag)',
    'archive.hourly.dewpoint_2m_avg_last_3h': 'Hourly 2 m dew point (3-hour rolling average)',
    'archive.hourly.dewpoint_2m_avg_last_6h': 'Hourly 2 m dew point (6-hour rolling average)',
    'archive.hourly.apparent_temperature': 'Hourly apparent temperature',
    'archive.hourly.apparent_temperature_lag_1h': 'Hourly apparent temperature (1-hour lag)',
    'archive.hourly.apparent_temperature_lag_3h': 'Hourly apparent temperature (3-hour lag)',
    'archive.hourly.apparent_temperature_avg_last_3h': 'Hourly apparent temperature (3-hour rolling average)',
    'archive.hourly.apparent_temperature_avg_last_6h': 'Hourly apparent temperature (6-hour rolling average)',
    'archive.hourly.precipitation': 'Hourly precipitation',
    'archive.hourly.precipitation_lag_1h': 'Hourly precipitation (1-hour lag)',
    'archive.hourly.precipitation_lag_3h': 'Hourly precipitation (3-hour lag)',
    'archive.hourly.precipitation_sum_last_3h': 'Hourly precipitation (3-hour rolling sum)',
    'archive.hourly.precipitation_sum_last_6h': 'Hourly precipitation (6-hour rolling sum)',
    'archive.hourly.rain': 'Hourly rain',
    'archive.hourly.rain_lag_1h': 'Hourly rain (1-hour lag)',
    'archive.hourly.rain_lag_3h': 'Hourly rain (3-hour lag)',
    'archive.hourly.rain_sum_last_3h': 'Hourly rain (3-hour rolling sum)',
    'archive.hourly.rain_sum_last_6h': 'Hourly rain (6-hour rolling sum)',
    'archive.hourly.snowfall': 'Hourly snowfall',
    'archive.hourly.snowfall_lag_1h': 'Hourly snowfall (1-hour lag)',
    'archive.hourly.snowfall_lag_3h': 'Hourly snowfall (3-hour lag)',
    'archive.hourly.snowfall_sum_last_3h': 'Hourly snowfall (3-hour rolling sum)',
    'archive.hourly.snowfall_sum_last_6h': 'Hourly snowfall (6-hour rolling sum)',
    'archive.hourly.snow_depth': 'Hourly snow depth',
    'archive.hourly.snow_depth_lag_1h': 'Hourly snow depth (1-hour lag)',
    'archive.hourly.snow_depth_lag_3h': 'Hourly snow depth (3-hour lag)',
    'archive.hourly.snow_depth_sum_last_3h': 'Hourly snow depth (3-hour rolling sum)',
    'archive.hourly.snow_depth_sum_last_6h': 'Hourly snow depth (6-hour rolling sum)',
    'archive.hourly.weathercode': 'Hourly weather code',
    'archive.hourly.weathercode_lag_1h': 'Hourly weather code (1-hour lag)',
    'archive.hourly.weathercode_lag_3h': 'Hourly weather code (3-hour lag)',
    'archive.hourly.weathercode_avg_last_3h': 'Hourly weather code (3-hour rolling average)',
    'archive.hourly.weathercode_avg_last_6h': 'Hourly weather code (6-hour rolling average)',
    'archive.hourly.pressure_msl': 'Hourly mean sea level pressure',
    'archive.hourly.pressure_msl_lag_1h': 'Hourly mean sea level pressure (1-hour lag)',
    'archive.hourly.pressure_msl_lag_3h': 'Hourly mean sea level pressure (3-hour lag)',
    'archive.hourly.pressure_msl_avg_last_3h': 'Hourly mean sea level pressure (3-hour rolling average)',
    'archive.hourly.pressure_msl_avg_last_6h': 'Hourly mean sea level pressure (6-hour rolling average)',
    'archive.hourly.surface_pressure': 'Hourly surface pressure',
    'archive.hourly.surface_pressure_lag_1h': 'Hourly surface pressure (1-hour lag)',
    'archive.hourly.surface_pressure_lag_3h': 'Hourly surface pressure (3-hour lag)',
    'archive.hourly.surface_pressure_avg_last_3h': 'Hourly surface pressure (3-hour rolling average)',
    'archive.hourly.surface_pressure_avg_last_6h': 'Hourly surface pressure (6-hour rolling average)',
    'archive.hourly.cloudcover': 'Hourly total cloud cover',
    'archive.hourly.cloudcover_lag_1h': 'Hourly total cloud cover (1-hour lag)',
    'archive.hourly.cloudcover_lag_3h': 'Hourly total cloud cover (3-hour lag)',
    'archive.hourly.cloudcover_avg_last_3h': 'Hourly total cloud cover (3-hour rolling average)',
    'archive.hourly.cloudcover_avg_last_6h': 'Hourly total cloud cover (6-hour rolling average)',
    'archive.hourly.cloudcover_low': 'Hourly low cloud cover',
    'archive.hourly.cloudcover_low_lag_1h': 'Hourly low cloud cover (1-hour lag)',
    'archive.hourly.cloudcover_low_lag_3h': 'Hourly low cloud cover (3-hour lag)',
    'archive.hourly.cloudcover_low_avg_last_3h': 'Hourly low cloud cover (3-hour rolling average)',
    'archive.hourly.cloudcover_low_avg_last_6h': 'Hourly low cloud cover (6-hour rolling average)',
    'archive.hourly.cloudcover_mid': 'Hourly mid cloud cover',
    'archive.hourly.cloudcover_mid_lag_1h': 'Hourly mid cloud cover (1-hour lag)',
    'archive.hourly.cloudcover_mid_lag_3h': 'Hourly mid cloud cover (3-hour lag)',
    'archive.hourly.cloudcover_mid_avg_last_3h': 'Hourly mid cloud cover (3-hour rolling average)',
    'archive.hourly.cloudcover_mid_avg_last_6h': 'Hourly mid cloud cover (6-hour rolling average)',
    'archive.hourly.cloudcover_high': 'Hourly high cloud cover',
    'archive.hourly.cloudcover_high_lag_1h': 'Hourly high cloud cover (1-hour lag)',
    'archive.hourly.cloudcover_high_lag_3h': 'Hourly high cloud cover (3-hour lag)',
    'archive.hourly.cloudcover_high_avg_last_3h': 'Hourly high cloud cover (3-hour rolling average)',
    'archive.hourly.cloudcover_high_avg_last_6h': 'Hourly high cloud cover (6-hour rolling average)',
    'archive.hourly.windspeed_10m': 'Hourly 10 m wind speed',
    'archive.hourly.windspeed_10m_lag_1h': 'Hourly 10 m wind speed (1-hour lag)',
    'archive.hourly.windspeed_10m_lag_3h': 'Hourly 10 m wind speed (3-hour lag)',
    'archive.hourly.windspeed_10m_avg_last_3h': 'Hourly 10 m wind speed (3-hour rolling average)',
    'archive.hourly.windspeed_10m_avg_last_6h': 'Hourly 10 m wind speed (6-hour rolling average)',
    'archive.hourly.winddirection_10m': 'Hourly 10 m wind direction',
    'archive.hourly.winddirection_10m_lag_1h': 'Hourly 10 m wind direction (1-hour lag)',
    'archive.hourly.winddirection_10m_lag_3h': 'Hourly 10 m wind direction (3-hour lag)',
    'archive.hourly.winddirection_10m_avg_last_3h': 'Hourly 10 m wind direction (3-hour rolling average)',
    'archive.hourly.winddirection_10m_avg_last_6h': 'Hourly 10 m wind direction (6-hour rolling average)',
    'archive.hourly.windgusts_10m': 'Hourly 10 m wind gusts',
    'archive.hourly.windgusts_10m_lag_1h': 'Hourly 10 m wind gusts (1-hour lag)',
    'archive.hourly.windgusts_10m_lag_3h': 'Hourly 10 m wind gusts (3-hour lag)',
    'archive.hourly.windgusts_10m_avg_last_3h': 'Hourly 10 m wind gusts (3-hour rolling average)',
    'archive.hourly.windgusts_10m_avg_last_6h': 'Hourly 10 m wind gusts (6-hour rolling average)',
    'archive.hourly.et0_fao_evapotranspiration': 'Hourly FAO reference evapotranspiration (ET0)',
    'archive.hourly.et0_fao_evapotranspiration_lag_1h': 'Hourly FAO reference evapotranspiration (ET0) (1-hour lag)',
    'archive.hourly.et0_fao_evapotranspiration_lag_3h': 'Hourly FAO reference evapotranspiration (ET0) (3-hour lag)',
    'archive.hourly.et0_fao_evapotranspiration_avg_last_3h': 'Hourly FAO reference evapotranspiration (ET0) (3-hour rolling average)',
    'archive.hourly.et0_fao_evapotranspiration_avg_last_6h': 'Hourly FAO reference evapotranspiration (ET0) (6-hour rolling average)',
    'archive.hourly.vapour_pressure_deficit': 'Hourly vapour pressure deficit',
    'archive.hourly.vapour_pressure_deficit_lag_1h': 'Hourly vapour pressure deficit (1-hour lag)',
    'archive.hourly.vapour_pressure_deficit_lag_3h': 'Hourly vapour pressure deficit (3-hour lag)',
    'archive.hourly.vapour_pressure_deficit_avg_last_3h': 'Hourly vapour pressure deficit (3-hour rolling average)',
    'archive.hourly.vapour_pressure_deficit_avg_last_6h': 'Hourly vapour pressure deficit (6-hour rolling average)',
    'archive.hourly.windspeed_100m': 'Hourly 100 m wind speed',
    'archive.hourly.windspeed_100m_lag_1h': 'Hourly 100 m wind speed (1-hour lag)',
    'archive.hourly.windspeed_100m_lag_3h': 'Hourly 100 m wind speed (3-hour lag)',
    'archive.hourly.windspeed_100m_avg_last_3h': 'Hourly 100 m wind speed (3-hour rolling average)',
    'archive.hourly.windspeed_100m_avg_last_6h': 'Hourly 100 m wind speed (6-hour rolling average)',
    'archive.hourly.winddirection_100m': 'Hourly 100 m wind direction',
    'archive.hourly.winddirection_100m_lag_1h': 'Hourly 100 m wind direction (1-hour lag)',
    'archive.hourly.winddirection_100m_lag_3h': 'Hourly 100 m wind direction (3-hour lag)',
    'archive.hourly.winddirection_100m_avg_last_3h': 'Hourly 100 m wind direction (3-hour rolling average)',
    'archive.hourly.winddirection_100m_avg_last_6h': 'Hourly 100 m wind direction (6-hour rolling average)',
    'archive.hourly.soil_temperature_0_to_7cm': 'Hourly soil temperature, 0 to 7 cm',
    'archive.hourly.soil_temperature_0_to_7cm_lag_1h': 'Hourly soil temperature, 0 to 7 cm (1-hour lag)',
    'archive.hourly.soil_temperature_0_to_7cm_lag_3h': 'Hourly soil temperature, 0 to 7 cm (3-hour lag)',
    'archive.hourly.soil_temperature_0_to_7cm_avg_last_3h': 'Hourly soil temperature, 0 to 7 cm (3-hour rolling average)',
    'archive.hourly.soil_temperature_0_to_7cm_avg_last_6h': 'Hourly soil temperature, 0 to 7 cm (6-hour rolling average)',
    'archive.hourly.soil_temperature_7_to_28cm': 'Hourly soil temperature, 7 to 28 cm',
    'archive.hourly.soil_temperature_7_to_28cm_lag_1h': 'Hourly soil temperature, 7 to 28 cm (1-hour lag)',
    'archive.hourly.soil_temperature_7_to_28cm_lag_3h': 'Hourly soil temperature, 7 to 28 cm (3-hour lag)',
    'archive.hourly.soil_temperature_7_to_28cm_avg_last_3h': 'Hourly soil temperature, 7 to 28 cm (3-hour rolling average)',
    'archive.hourly.soil_temperature_7_to_28cm_avg_last_6h': 'Hourly soil temperature, 7 to 28 cm (6-hour rolling average)',
    'archive.hourly.soil_temperature_28_to_100cm': 'Hourly soil temperature, 28 to 100 cm',
    'archive.hourly.soil_temperature_28_to_100cm_lag_1h': 'Hourly soil temperature, 28 to 100 cm (1-hour lag)',
    'archive.hourly.soil_temperature_28_to_100cm_lag_3h': 'Hourly soil temperature, 28 to 100 cm (3-hour lag)',
    'archive.hourly.soil_temperature_28_to_100cm_avg_last_3h': 'Hourly soil temperature, 28 to 100 cm (3-hour rolling average)',
    'archive.hourly.soil_temperature_28_to_100cm_avg_last_6h': 'Hourly soil temperature, 28 to 100 cm (6-hour rolling average)',
    'archive.hourly.soil_temperature_100_to_255cm': 'Hourly soil temperature, 100 to 255 cm',
    'archive.hourly.soil_temperature_100_to_255cm_lag_1h': 'Hourly soil temperature, 100 to 255 cm (1-hour lag)',
    'archive.hourly.soil_temperature_100_to_255cm_lag_3h': 'Hourly soil temperature, 100 to 255 cm (3-hour lag)',
    'archive.hourly.soil_temperature_100_to_255cm_avg_last_3h': 'Hourly soil temperature, 100 to 255 cm (3-hour rolling average)',
    'archive.hourly.soil_temperature_100_to_255cm_avg_last_6h': 'Hourly soil temperature, 100 to 255 cm (6-hour rolling average)',
    'archive.hourly.soil_moisture_0_to_7cm': 'Hourly soil moisture, 0 to 7 cm',
    'archive.hourly.soil_moisture_0_to_7cm_lag_1h': 'Hourly soil moisture, 0 to 7 cm (1-hour lag)',
    'archive.hourly.soil_moisture_0_to_7cm_lag_3h': 'Hourly soil moisture, 0 to 7 cm (3-hour lag)',
    'archive.hourly.soil_moisture_0_to_7cm_avg_last_3h': 'Hourly soil moisture, 0 to 7 cm (3-hour rolling average)',
    'archive.hourly.soil_moisture_0_to_7cm_avg_last_6h': 'Hourly soil moisture, 0 to 7 cm (6-hour rolling average)',
    'archive.hourly.soil_moisture_7_to_28cm': 'Hourly soil moisture, 7 to 28 cm',
    'archive.hourly.soil_moisture_7_to_28cm_lag_1h': 'Hourly soil moisture, 7 to 28 cm (1-hour lag)',
    'archive.hourly.soil_moisture_7_to_28cm_lag_3h': 'Hourly soil moisture, 7 to 28 cm (3-hour lag)',
    'archive.hourly.soil_moisture_7_to_28cm_avg_last_3h': 'Hourly soil moisture, 7 to 28 cm (3-hour rolling average)',
    'archive.hourly.soil_moisture_7_to_28cm_avg_last_6h': 'Hourly soil moisture, 7 to 28 cm (6-hour rolling average)',
    'archive.hourly.soil_moisture_28_to_100cm': 'Hourly soil moisture, 28 to 100 cm',
    'archive.hourly.soil_moisture_28_to_100cm_lag_1h': 'Hourly soil moisture, 28 to 100 cm (1-hour lag)',
    'archive.hourly.soil_moisture_28_to_100cm_lag_3h': 'Hourly soil moisture, 28 to 100 cm (3-hour lag)',
    'archive.hourly.soil_moisture_28_to_100cm_avg_last_3h': 'Hourly soil moisture, 28 to 100 cm (3-hour rolling average)',
    'archive.hourly.soil_moisture_28_to_100cm_avg_last_6h': 'Hourly soil moisture, 28 to 100 cm (6-hour rolling average)',
    'archive.hourly.soil_moisture_100_to_255cm': 'Hourly soil moisture, 100 to 255 cm',
    'archive.hourly.soil_moisture_100_to_255cm_lag_1h': 'Hourly soil moisture, 100 to 255 cm (1-hour lag)',
    'archive.hourly.soil_moisture_100_to_255cm_lag_3h': 'Hourly soil moisture, 100 to 255 cm (3-hour lag)',
    'archive.hourly.soil_moisture_100_to_255cm_avg_last_3h': 'Hourly soil moisture, 100 to 255 cm (3-hour rolling average)',
    'archive.hourly.soil_moisture_100_to_255cm_avg_last_6h': 'Hourly soil moisture, 100 to 255 cm (6-hour rolling average)',
    'archive.daily.weathercode': 'Daily weather code',
    'archive.daily.temperature_2m_max': 'Daily Temperature 2m max',
    'archive.daily.temperature_2m_min': 'Daily Temperature 2m min',
    'archive.daily.apparent_temperature_max': 'Daily Apparent temperature max',
    'archive.daily.apparent_temperature_min': 'Daily Apparent temperature min',
    'archive.daily.sunrise': 'Daily sunrise',
    'archive.daily.sunset': 'Daily sunset',
    'archive.daily.daylight_duration': 'Daily daylight duration',
    'archive.daily.sunshine_duration': 'Daily sunshine duration',
    'archive.daily.precipitation_sum': 'Daily precipitation sum',
    'archive.daily.rain_sum': 'Daily rain sum',
    'archive.daily.snowfall_sum': 'Daily snowfall sum',
    'archive.daily.precipitation_hours': 'Daily Precipitation hours',
    'archive.daily.windspeed_10m_max': 'Daily Windspeed 10m max',
    'archive.daily.windgusts_10m_max': 'Daily Windgusts 10m max',
    'archive.daily.winddirection_10m_dominant': 'Daily dominant 10 m wind direction',
    'archive.daily.shortwave_radiation_sum': 'Daily shortwave radiation sum',
    'archive.daily.et0_fao_evapotranspiration': 'Daily FAO reference evapotranspiration (ET0)',
    'past_accidents_50m_1y': 'Past accidents within 50 m in the last year',
    'past_accidents_100m_1y': 'Past accidents within 100 m in the last year',
    'past_accidents_200m_1y': 'Past accidents within 200 m in the last year',
    'past_accidents_50m_2y': 'Past accidents within 50 m in the last 2 years',
    'past_accidents_100m_2y': 'Past accidents within 100 m in the last 2 years',
    'past_accidents_200m_2y': 'Past accidents within 200 m in the last 2 years',
    'past_accidents_50m_3y': 'Past accidents within 50 m in the last 3 years',
    'past_accidents_100m_3y': 'Past accidents within 100 m in the last 3 years',
    'past_accidents_200m_3y': 'Past accidents within 200 m in the last 3 years',
    'nearest_pub_road_lat': 'Nearest pub road latitude',
    'nearest_pub_road_lon': 'Nearest pub road longitude',
    'nearest_pub_road_crow_m': 'Crow-flight distance to nearest pub road (m)',
    'nearest_pub_road_m': 'Road-network distance to nearest pub (m)',
    'n_pubs_100m': 'Number of pubs within 100 m',
    'nearest_school_road_urn': 'Nearest school road URN',
    'nearest_school_road_phase': 'Nearest school road phase',
    'nearest_school_road_E': 'Nearest school road easting',
    'nearest_school_road_N': 'Nearest school road northing',
    'nearest_school_road_crow_m': 'Crow-flight distance to nearest school road (m)',
    'nearest_school_road_m': 'Road-network distance to nearest school (m)',
    'n_schools_100m': 'Number of schools within 100 m',
    'nearest_overground_road_E': 'Nearest overground station road easting',
    'nearest_overground_road_N': 'Nearest overground station road northing',
    'nearest_overground_road_crow_m': 'Crow-flight distance to nearest overground station road (m)',
    'nearest_overground_road_m': 'Road-network distance to nearest overground station (m)',
    'n_overground_100m': 'Number of overground stations within 100 m',
    'nearest_underground_road_E': 'Nearest underground station road easting',
    'nearest_underground_road_N': 'Nearest underground station road northing',
    'nearest_underground_road_crow_m': 'Crow-flight distance to nearest underground station road (m)',
    'nearest_underground_road_m': 'Road-network distance to nearest underground station (m)',
    'n_underground_100m': 'Number of underground stations within 100 m',
    'nearest_signal_road_E': 'Nearest signal road easting',
    'nearest_signal_road_N': 'Nearest signal road northing',
    'nearest_signal_road_crow_m': 'Crow-flight distance to nearest signal road (m)',
    'nearest_signal_road_m': 'Road-network distance to nearest signal (m)',
    'n_signals_100m': 'Number of signals within 100 m',
    'nearest_bus_stop_road_name': 'Nearest bus stop road name',
    'nearest_bus_stop_road_E': 'Nearest bus stop road easting',
    'nearest_bus_stop_road_N': 'Nearest bus stop road northing',
    'nearest_bus_stop_road_crow_m': 'Crow-flight distance to nearest bus stop road (m)',
    'nearest_bus_stop_road_m': 'Road-network distance to nearest bus stop (m)',
    'n_bus_stops_100m': 'Number of bus stops within 100 m',
    'UID': 'Unique ID',
    'HS_bin': 'HS bin',
    'IMDRank': 'Index of Multiple Deprivation rank',
    'IMDDecile': 'Index of Multiple Deprivation decile',
}

def get_column_label(column: str) -> str:
    return COLUMN_LABELS.get(column, column)

def infer_numeric_columns(df: pd.DataFrame, requested: Optional[Sequence[str]] = None) -> List[str]:
    if requested:
        missing = [c for c in requested if c not in df.columns]
        if missing:
            raise ValueError(f"These columns were not found in the CSV: {missing}")
        numeric_requested = [c for c in requested if pd.api.types.is_numeric_dtype(df[c])]
        skipped = [c for c in requested if c not in numeric_requested]
        if skipped:
            print(f"Skipping non-numeric columns: {skipped}")
        return numeric_requested

    return df.select_dtypes(include=[np.number]).columns.tolist()


def _finite_values(series: pd.Series) -> np.ndarray:
    values = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    return values[np.isfinite(values)]


def _safe_quantile(values: np.ndarray, q: float) -> float:
    if values.size == 0:
        return float("nan")
    return float(np.quantile(values, q))


def choose_scale_and_bins(
    values: np.ndarray,
    skew_threshold: float,
    bins: int,
) -> Tuple[str, np.ndarray, Optional[float]]:
    """Return (scale, bin_edges, linthresh_for_symlog)."""
    if values.size == 0:
        return "linear", np.linspace(0, 1, max(bins, 2)), None

    skew = float(pd.Series(values).skew()) if values.size >= 3 else 0.0
    finite = values[np.isfinite(values)]
    finite_nonzero = finite[finite != 0]

    positive = finite[finite > 0]
    negative = finite[finite < 0]

    if abs(skew) < skew_threshold:
        return "linear", np.histogram_bin_edges(values, bins=bins), None

    if negative.size == 0 and positive.size > 0:
        min_pos = max(np.min(positive), np.finfo(float).tiny)
        max_pos = np.max(positive)
        if np.isclose(min_pos, max_pos):
            return "linear", np.histogram_bin_edges(values, bins=bins), None
        edges = np.logspace(np.log10(min_pos), np.log10(max_pos), bins + 1)
        return "log", edges, None

    abs_nonzero = np.abs(finite_nonzero)
    if abs_nonzero.size > 0:
        linthresh = float(np.quantile(abs_nonzero, 0.1))
        if not np.isfinite(linthresh) or linthresh <= 0:
            linthresh = 1.0
    else:
        linthresh = 1.0
    return "symlog", np.histogram_bin_edges(values, bins=bins), linthresh


def compute_xlim(values: np.ndarray, clip_quantiles: Tuple[float, float]) -> Optional[Tuple[float, float]]:
    low_q, high_q = clip_quantiles
    if low_q <= 0 and high_q >= 1:
        return None
    if values.size == 0:
        return None
    lo = _safe_quantile(values, low_q)
    hi = _safe_quantile(values, high_q)
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return None
    return lo, hi


# replace plot_feature(...) with this version

def plot_feature(
    ax: plt.Axes,
    values: np.ndarray,
    name: str,
    skew_threshold: float,
    bins: int,
    clip_quantiles: Tuple[float, float],
) -> None:
    label = get_column_label(name)

    if values.size == 0:
        ax.text(0.5, 0.5, "No valid numeric data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(label)
        ax.set_axis_off()
        return

    scale, edges, symlog_linthresh = choose_scale_and_bins(values, skew_threshold, bins)
    skew = float(pd.Series(values).skew()) if values.size >= 3 else 0.0
    mean = float(np.mean(values))
    median = float(np.median(values))

    counts, _, _ = ax.hist(values, bins=edges, edgecolor="black", alpha=0.85)
    y_scale, y_linthresh = choose_count_scale(counts, skew_threshold)

    ax.set_title(label)
    ax.set_ylabel("Occurrences")
    ax.set_xlabel(f"{label} ({scale} scale)" if scale != "linear" else label)

    if scale == "log":
        ax.set_xscale("log")
    elif scale == "symlog":
        ax.set_xscale("symlog", linthresh=symlog_linthresh or 1.0)

    if y_scale == "log":
        ax.set_yscale("log")
    elif y_scale == "symlog":
        ax.set_yscale("symlog", linthresh=y_linthresh or 1.0)

    if scale == "linear":
        xlim = compute_xlim(values, clip_quantiles)
        if xlim is not None:
            ax.set_xlim(*xlim)

    annotation = f"skew={skew:.2f}\nmean={mean:.2g}\nmedian={median:.2g}"
    if scale != "linear":
        annotation += f"\nX={scale}"
    if y_scale != "linear":
        annotation += f"\nY={y_scale}"
    ax.text(
        0.98,
        0.98,
        annotation,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=9,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.75, edgecolor="none"),
    )
    ax.grid(True, alpha=0.2)

def chunked(items: Sequence[str], size: int) -> Iterable[Sequence[str]]:
    for i in range(0, len(items), size):
        yield items[i : i + size]

def choose_count_scale(counts: np.ndarray, skew_threshold: float) -> Tuple[str, Optional[float]]:
    counts = counts[np.isfinite(counts) & (counts > 0)]
    if counts.size == 0:
        return "linear", None

    skew = float(pd.Series(counts).skew()) if counts.size >= 3 else 0.0
    if abs(skew) < skew_threshold:
        return "linear", None

    if np.min(counts) > 0:
        return "log", None

    linthresh = float(np.quantile(counts, 0.1))
    if not np.isfinite(linthresh) or linthresh <= 0:
        linthresh = 1.0
    return "symlog", linthresh


def create_distribution_sheets(
    df: pd.DataFrame,
    columns: Optional[Sequence[str]],
    output_dir: Path,
    skew_threshold: float,
    bins: int,
    dpi: int,
    fig_width: float,
    fig_height: float,
    clip_quantiles: Tuple[float, float],
) -> List[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)

    selected_cols = infer_numeric_columns(df, columns)
    if not selected_cols:
        raise ValueError("No numeric columns were found to plot.")

    saved_files: List[Path] = []
    cols_per_page = 6  # 2 rows x 3 columns

    for page_idx, cols in enumerate(chunked(selected_cols, cols_per_page), start=1):
        fig, axes = plt.subplots(2, 3, figsize=(fig_width, fig_height), constrained_layout=True)
        axes_flat = axes.flatten()

        for ax, col in zip(axes_flat, cols):
            values = _finite_values(df[col])
            plot_feature(ax, values, col, skew_threshold, bins, clip_quantiles)

        for ax in axes_flat[len(cols) :]:
            ax.set_visible(False)

        fig.suptitle("Feature Distributions", fontsize=16, y=1.02)
        out_path = output_dir / f"feature_distributions_page_{page_idx:02d}.png"
        fig.savefig(out_path, dpi=dpi, bbox_inches="tight")
        plt.close(fig)
        saved_files.append(out_path)

    return saved_files


# =========================
# RUN
# =========================
df = pd.read_csv(CSV_PATH)
saved = create_distribution_sheets(
    df=df,
    columns=COLUMNS,
    output_dir=Path(OUTPUT_DIR),
    skew_threshold=SKEW_THRESHOLD,
    bins=BINS,
    dpi=DPI,
    fig_width=FIGSIZE[0],
    fig_height=FIGSIZE[1],
    clip_quantiles=CLIP_QUANTILES,
)

print("Saved files:")
for path in saved:
    print(f"- {path}")
